In [2]:
import pandas as pd
import os

seasons = [
    "E0", "E0 (1)", "E0 (2)", "E0 (3)", "E0 (4)",
    "E0 (5)", "E0 (6)", "E0 (7)", "E0 (8)", "E0 (9)"
]

all_seasons = []

for season in seasons:
    df = pd.read_csv(f"../data/{season}.csv", encoding="latin1")

    df["Season"] = season

    all_seasons.append(df)

df = pd.concat(all_seasons, ignore_index=True)

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

print(df.shape)
print(df.head())
print(df["Season"].value_counts().sort_index())
print(df["Season"].nunique())

(3800, 155)
  ï»¿Div       Date Time     HomeTeam        AwayTeam  FTHG  FTAG FTR  HTHG  \
0    NaN 2015-08-08  NaN  Bournemouth     Aston Villa     0     1   A     0   
1    NaN 2015-08-08  NaN      Norwich  Crystal Palace     1     3   A     0   
2    NaN 2015-08-08  NaN   Man United       Tottenham     1     0   H     1   
3    NaN 2015-08-08  NaN      Chelsea         Swansea     2     2   D     2   
4    NaN 2015-08-08  NaN      Everton         Watford     2     2   D     0   

   HTAG  ... BbAv<2.5  BbAH  BbAHh  BbMxAHH  BbAvAHH  BbMxAHA  BbAvAHA   LBH  \
0     0  ...     1.79  26.0   -0.5     1.98     1.93     1.99     1.92  2.05   
1     1  ...     1.67  27.0    0.0     1.83     1.78     2.17     2.08  2.62   
2     0  ...     2.01  26.0   -1.0     2.20     2.09     1.82     1.78  1.67   
3     1  ...     1.99  27.0   -1.5     2.24     2.16     1.80     1.73  1.40   
4     1  ...     1.96  26.0   -1.0     2.28     2.18     1.76     1.71  1.75   

   LBD   LBA  
0  3.3   4.0  
1 

We now have all 10 seasons in a single dataframe. Let's find the attacking and defensive strengths for the 10 best teams (percentage above/below league average).

In [3]:
league_home_avg = df["FTHG"].mean()
league_away_avg = df["FTAG"].mean()

print("Average home goals:", league_home_avg)
print("Average away goals:", league_away_avg)

teams = sorted(set(df["HomeTeam"]) | set(df["AwayTeam"]))

team_stats = []

for team in teams:

    home_matches = df[df["HomeTeam"] == team]
    away_matches = df[df["AwayTeam"] == team]

    goals_scored = (
        home_matches["FTHG"].sum() +
        away_matches["FTAG"].sum()
    )

    goals_conceded = (
        home_matches["FTAG"].sum() +
        away_matches["FTHG"].sum()
    )

    matches_played = len(home_matches) + len(away_matches)

    avg_scored = goals_scored / matches_played
    avg_conceded = goals_conceded / matches_played

    attack_strength = avg_scored / ((league_home_avg + league_away_avg) / 2)
    defence_strength = avg_conceded / ((league_home_avg + league_away_avg) / 2)

    team_stats.append({
        "Team": team,
        "Matches": matches_played,
        "Goals Scored": goals_scored,
        "Goals Conceded": goals_conceded,
        "Attack Strength": attack_strength,
        "Defence Strength": defence_strength
    })

team_stats = pd.DataFrame(team_stats)

print(team_stats.sort_values("Attack Strength", ascending=False).head(10))

Average home goals: 1.551842105263158
Average away goals: 1.2781578947368422
          Team  Matches  Goals Scored  Goals Conceded  Attack Strength  \
18    Man City      380           898             334         1.670076   
16   Liverpool      380           808             382         1.502697   
0      Arsenal      380           709             423         1.318579   
29   Tottenham      380           702             457         1.305561   
7      Chelsea      380           651             439         1.210712   
19  Man United      380           591             438         1.099126   
3    Brentford      152           228             224         1.060071   
15   Leicester      342           504             505         1.041473   
32    West Ham      380           531             589         0.987540   
21   Newcastle      342           474             484         0.979481   

    Defence Strength  
18          0.621164  
16          0.710433  
0           0.786684  
29          0.84

Our next goal is to predict the expected goals in a fixture.

In [4]:
def predict_expected_goals(home_team, away_team):

    home_attack = team_stats.loc[
        team_stats["Team"] == home_team, "Attack Strength"
    ].iloc[0]

    home_defence = team_stats.loc[
        team_stats["Team"] == home_team, "Defence Strength"
    ].iloc[0]

    away_attack = team_stats.loc[
        team_stats["Team"] == away_team, "Attack Strength"
    ].iloc[0]

    away_defence = team_stats.loc[
        team_stats["Team"] == away_team, "Defence Strength"
    ].iloc[0]

    expected_home = league_home_avg * home_attack * away_defence
    expected_away = league_away_avg * away_attack * home_defence

    return expected_home, expected_away


print(predict_expected_goals("Man City", "Arsenal"))

(np.float64(2.0388447735987603), np.float64(1.0468805615146939))


We can now turn these expected goals into probabilities using the Poisson distribution.

In [5]:
from scipy.stats import poisson
import numpy as np

home_lambda, away_lambda = predict_expected_goals(
    "Man City", "Arsenal"
)

goals = np.arange(0, 11)

home_probs = poisson.pmf(goals, home_lambda)
away_probs = poisson.pmf(goals, away_lambda)

score_matrix = np.outer(home_probs, away_probs)

print(np.round(score_matrix, 3))

home_win = np.tril(score_matrix, -1).sum()
draw = np.trace(score_matrix)
away_win = np.triu(score_matrix, 1).sum()

print(f"Home win: {home_win:.2%}")
print(f"Draw: {draw:.2%}")
print(f"Away win: {away_win:.2%}")

[[0.046 0.048 0.025 0.009 0.002 0.    0.    0.    0.    0.    0.   ]
 [0.093 0.098 0.051 0.018 0.005 0.001 0.    0.    0.    0.    0.   ]
 [0.095 0.099 0.052 0.018 0.005 0.001 0.    0.    0.    0.    0.   ]
 [0.065 0.068 0.035 0.012 0.003 0.001 0.    0.    0.    0.    0.   ]
 [0.033 0.034 0.018 0.006 0.002 0.    0.    0.    0.    0.    0.   ]
 [0.013 0.014 0.007 0.003 0.001 0.    0.    0.    0.    0.    0.   ]
 [0.005 0.005 0.002 0.001 0.    0.    0.    0.    0.    0.    0.   ]
 [0.001 0.001 0.001 0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.   ]]
Home win: 60.28%
Draw: 20.94%
Away win: 18.78%


Next, I want to also introduce recent performances as a factor which influences our probabilities. As I believe this is extremely important as many teams may not have historically performed in the past, therefore they will be vastly underestimated despite great improvement in recent times.

In [6]:
home = df[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].copy()

home.columns = [
    "Date", "Team", "Opponent",
    "GoalsScored", "GoalsConceded", "Result"
]

home["Venue"] = "Home"

away = df[["Date", "AwayTeam", "HomeTeam", "FTAG", "FTHG", "FTR"]].copy()

away.columns = [
    "Date", "Team", "Opponent",
    "GoalsScored", "GoalsConceded", "Result"
]

away["Venue"] = "Away"

team_matches = pd.concat([home, away], ignore_index=True)

team_matches = team_matches.sort_values(
    ["Team", "Date"]
).reset_index(drop=True)

print(team_matches.head(10))

        Date     Team        Opponent  GoalsScored  GoalsConceded Result Venue
0 2015-08-09  Arsenal        West Ham            0              2      A  Home
1 2015-08-16  Arsenal  Crystal Palace            2              1      A  Away
2 2015-08-24  Arsenal       Liverpool            0              0      D  Home
3 2015-08-29  Arsenal       Newcastle            1              0      A  Away
4 2015-09-12  Arsenal           Stoke            2              0      H  Home
5 2015-09-19  Arsenal         Chelsea            0              2      H  Away
6 2015-09-26  Arsenal       Leicester            5              2      A  Away
7 2015-10-04  Arsenal      Man United            3              0      H  Home
8 2015-10-17  Arsenal         Watford            3              0      A  Away
9 2015-10-24  Arsenal         Everton            2              1      H  Home


In [7]:
def calculate_points(row):

    if row["GoalsScored"] > row["GoalsConceded"]:
        return 3

    elif row["GoalsScored"] == row["GoalsConceded"]:
        return 1

    else:
        return 0


team_matches["Points"] = team_matches.apply(
    calculate_points, axis=1
)

columns = ["GoalsScored", "GoalsConceded", "Points"]

for column in columns:

    team_matches[f"Recent{column}"] = (
        team_matches.groupby("Team")[column]
        .transform(lambda x: x.shift(1).rolling(5).mean())
    )

In [8]:
print(
    team_matches[
        ["Date", "Team", "Opponent",
         "GoalsScored", "Points",
         "RecentGoalsScored",
         "RecentGoalsConceded",
         "RecentPoints"]
    ].tail(15)
)

           Date    Team        Opponent  GoalsScored  Points  \
7585 2025-02-01  Wolves     Aston Villa            2       3   
7586 2025-02-16  Wolves       Liverpool            1       0   
7587 2025-02-22  Wolves     Bournemouth            1       3   
7588 2025-02-25  Wolves          Fulham            1       0   
7589 2025-03-08  Wolves         Everton            1       1   
7590 2025-03-15  Wolves     Southampton            2       3   
7591 2025-04-01  Wolves        West Ham            1       3   
7592 2025-04-05  Wolves         Ipswich            2       3   
7593 2025-04-13  Wolves       Tottenham            4       3   
7594 2025-04-20  Wolves      Man United            1       3   
7595 2025-04-26  Wolves       Leicester            3       3   
7596 2025-05-02  Wolves        Man City            0       0   
7597 2025-05-10  Wolves        Brighton            0       0   
7598 2025-05-20  Wolves  Crystal Palace            2       0   
7599 2025-05-25  Wolves       Brentford 

Let's implement recent-form features for both teams on every fixture.

In [9]:
all_seasons = []

for season in seasons:

    season_df = pd.read_csv(
        f"../data/{season}.csv",
        encoding="latin1"
    )

    season_df["Date"] = pd.to_datetime(
        season_df["Date"],
        dayfirst=True,
        errors="coerce"
    )

    season_df["Season"] = season

    all_seasons.append(season_df)

df = pd.concat(all_seasons, ignore_index=True)

df = df.sort_values("Date").reset_index(drop=True)

form_columns = [
    "RecentGoalsScored",
    "RecentGoalsConceded",
    "RecentPoints"
]

home_form = team_matches[
    ["Date", "Team"] + form_columns
].copy()

away_form = team_matches[
    ["Date", "Team"] + form_columns
].copy()

home_form = home_form.rename(columns={
    "Team": "HomeTeam",
    "RecentGoalsScored": "HomeRecentGoalsScored",
    "RecentGoalsConceded": "HomeRecentGoalsConceded",
    "RecentPoints": "HomeRecentPoints"
})

away_form = away_form.rename(columns={
    "Team": "AwayTeam",
    "RecentGoalsScored": "AwayRecentGoalsScored",
    "RecentGoalsConceded": "AwayRecentGoalsConceded",
    "RecentPoints": "AwayRecentPoints"
})

model_df = df.merge(
    home_form,
    on=["Date", "HomeTeam"],
    how="left"
)

model_df = model_df.merge(
    away_form,
    on=["Date", "AwayTeam"],
    how="left"
)

/var/folders/tr/k8s41dv95xl1ybw_ty64tp680000gn/T/ipykernel_95950/106185474.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  season_df["Date"] = pd.to_datetime(


In [10]:
print("Original dataset:", df.shape)
print("Model dataset:", model_df.shape)

print(model_df[[
    "Date",
    "HomeTeam",
    "AwayTeam",
    "HomeRecentGoalsScored",
    "HomeRecentPoints",
    "AwayRecentGoalsScored",
    "AwayRecentPoints"
]].tail())

Original dataset: (3800, 155)
Model dataset: (3800, 161)
           Date     HomeTeam   AwayTeam  HomeRecentGoalsScored  \
3795 2025-05-25       Fulham   Man City                    1.4   
3796 2025-05-25  Bournemouth  Leicester                    0.8   
3797 2025-05-25    Tottenham   Brighton                    0.6   
3798 2025-05-25      Ipswich   West Ham                    0.4   
3799 2025-05-25  Southampton    Arsenal                    0.4   

      HomeRecentPoints  AwayRecentGoalsScored  AwayRecentPoints  
3795               1.2                    1.6               2.6  
3796               1.0                    1.2               1.4  
3797               0.2                    2.2               2.0  
3798               0.2                    1.4               1.0  
3799               0.4                    2.0               1.6  


In [11]:
features = [
    "HomeRecentGoalsScored",
    "HomeRecentGoalsConceded",
    "HomeRecentPoints",
    "AwayRecentGoalsScored",
    "AwayRecentGoalsConceded",
    "AwayRecentPoints"
]

X = model_df[features]
y = model_df["FTR"]

valid_rows = X.notna().all(axis=1)

X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

print("Remaining matches:", len(X))
print(y.value_counts())

dates = model_df.loc[valid_rows, "Date"]

train_mask = dates < "2023-08-01"

val_mask = (
    (dates >= "2023-08-01") &
    (dates < "2024-08-01")
)

test_mask = dates >= "2024-08-01"

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

Remaining matches: 3310
FTR
H    1464
A    1079
D     767
Name: count, dtype: int64
Training: 2560
Validation: 375
Testing: 375
